In [40]:
import pandas as pd
import numpy as np

# 물동량 예측 파일 EDA

In [41]:
CSV_PATH = './D003.csv'

df = pd.read_csv(CSV_PATH, encoding ='cp949')

In [42]:
df.head()

,연도,월,연월,수출입구분,수출입구분설명,물동량
0,2003,2,2003-02,OT,수출환적,161074.50
1,2003,11,2003-11,IT,수입환적,170733.50
2,2006,1,2006-01,OT,수출환적,203348.25
3,2018,1,2018-01,OO,수출,413797.75
4,2013,6,2013-06,OO,수출,389650.25


In [43]:
df.shape

(1376, 6)

In [44]:
df.columns

Index(['연도', '월', '연월', '수출입구분', '수출입구분설명', '물동량'], dtype='str')

In [ ]:
df.dtypes
# 연월이 문자열로 있으면 시간계산 변화율이 어려우니 문자열로 바꿔야함

연도           int64
월            int64
연월             str
수출입구분          str
수출입구분설명        str
물동량        float64
dtype: object

In [46]:
df.info

<bound method DataFrame.info of         연도   월       연월 수출입구분 수출입구분설명        물동량
0     2003   2  2003-02    OT    수출환적  161074.50
1     2003  11  2003-11    IT    수입환적  170733.50
2     2006   1  2006-01    OT    수출환적  203348.25
3     2018   1  2018-01    OO      수출  413797.75
4     2013   6  2013-06    OO      수출  389650.25
...    ...  ..      ...   ...     ...        ...
1371  1996   1  1996-01    OO      수출  158518.00
1372  2001   6  2001-06    OO      수출  215325.25
1373  2001   9  2001-09    OT    수출환적  127883.00
1374  2004   1  2004-01    IT    수입환적  181137.50
1375  2005   3  2005-03    OT    수출환적  221527.25

[1376 rows x 6 columns]>

In [47]:
type(pd.DataFrame)

type

In [48]:
pd.DataFrame({
    '고유값 수': df.nunique(dropna=True)})

,고유값 수
연도,34
월,12
연월,398
수출입구분,4
수출입구분설명,4
물동량,1374


# 1 장기 성장 확인하기

In [49]:
월별_물동량 = pd.DataFrame(
    {'연도' : df['연도'],
     '월': df['월'],
     '물동량': df['물동량']})

월별_물동량 = 월별_물동량.sort_values(
    by=['연도','월']
)

월별_물동량

,연도,월,물동량
602,1992,1,104567.00
864,1992,1,92005.00
491,1992,2,91669.00
735,1992,2,103903.00
282,1992,3,137082.00
...,...,...,...
1232,2025,1,468006.50
50,2025,2,437906.75
207,2025,2,599626.75
858,2025,2,392441.25


In [50]:
연도별_물동량 = (
    월별_물동량
    .groupby('연도',as_index=False)['물동량'].sum()
)

연도별_물동량

,연도,물동량
0,1992,2635374.00
1,1993,2941235.00
2,1994,3582517.00
3,1995,4090333.75
4,1996,4362035.00
5,1997,4811161.50
6,1998,5311311.50
7,1999,5660882.00
8,2000,6382317.75
9,2001,8072801.75


# 2 계절성 확인하기

In [51]:
df_2425 = df.copy()
df_2425 = df[(df['연도'] >= 2024) & (df['연도'] <= 2025)]

df_2425 = df_2425.sort_values(
    by=['연도', '월'],
    ascending=[True, True]
)

df_2425.head()

,연도,월,연월,수출입구분,수출입구분설명,물동량
769,2024,1,2024-01,II,수입,428462.25
936,2024,1,2024-01,IT,수입환적,558618.00
1113,2024,1,2024-01,OT,수출환적,547360.25
1189,2024,1,2024-01,OO,수출,457956.00
234,2024,2,2024-02,IT,수입환적,525700.25


In [52]:
monthly_total = (
    df_2425
    .groupby(['연도', '월'], as_index=False)['물동량']
    .sum()
)

monthly_total

,연도,월,물동량
0,2024,1,1992396.50
1,2024,2,1879271.50
2,2024,3,2143065.75
3,2024,4,2046147.00
4,2024,5,2098141.00
5,2024,6,2090886.75
6,2024,7,2107241.75
7,2024,8,2052806.00
8,2024,9,1877152.00
9,2024,10,2049390.75


In [53]:
monthly_total = monthly_total.sort_values(by=['연도','월'])

In [54]:
# 지난 달 대비 현재 월의 성장률
monthly_total['지난달_물동량'] = monthly_total['물동량'].shift(1)

monthly_total['물동량 변화율'] = (
    (monthly_total['물동량'] - monthly_total['지난달_물동량']) / monthly_total['지난달_물동량']) * 100


monthly_total

,연도,월,물동량,지난달_물동량,물동량 변화율
0,2024,1,1992396.50,NaN,NaN
1,2024,2,1879271.50,1992396.50,-5.677836
2,2024,3,2143065.75,1879271.50,14.037048
3,2024,4,2046147.00,2143065.75,-4.522435
4,2024,5,2098141.00,2046147.00,2.541069
5,2024,6,2090886.75,2098141.00,-0.345747
6,2024,7,2107241.75,2090886.75,0.782204
7,2024,8,2052806.00,2107241.75,-2.583270
8,2024,9,1877152.00,2052806.00,-8.556775
9,2024,10,2049390.75,1877152.00,9.175536


In [55]:
# 연간 월평균 물동량 기준으로 월별 증가율 확인

annual_monthly_mean = monthly_total['물동량'].mean()


monthly_total['평균대비(%)'] = (
    (monthly_total['물동량'] -  annual_monthly_mean) / annual_monthly_mean
) * 100

monthly_total

,연도,월,물동량,지난달_물동량,물동량 변화율,평균대비(%)
0,2024,1,1992396.50,NaN,NaN,-2.099202
1,2024,2,1879271.50,1992396.50,-5.677836,-7.657848
2,2024,3,2143065.75,1879271.50,14.037048,5.304264
3,2024,4,2046147.00,2143065.75,-4.522435,0.541947
4,2024,5,2098141.00,2046147.00,2.541069,3.096787
5,2024,6,2090886.75,2098141.00,-0.345747,2.740334
6,2024,7,2107241.75,2090886.75,0.782204,3.543973
7,2024,8,2052806.00,2107241.75,-2.583270,0.869152
8,2024,9,1877152.00,2052806.00,-8.556775,-7.761995
9,2024,10,2049390.75,1877152.00,9.175536,0.701336


# 3 환적 구조

In [59]:
수출 = (
    df[df['수출입구분설명'] == '수출']
    .sort_values(['연도','월'])
    .reset_index(drop=True)
)

수입 = (
    df[df['수출입구분설명'] == '수입']
    .sort_values(['연도','월'])
    .reset_index(drop=True)
)

수출환적 = (
    df[df['수출입구분설명'] == '수출환적']
    .sort_values(['연도','월'])
    .reset_index(drop=True)
)

수입환적 = (
    df[df['수출입구분설명'] == '수입환적']
    .sort_values(['연도','월'])
    .reset_index(drop=True)
)

수출.head
수입.head
수출환적.head
수입환적.head

<bound method NDFrame.head of        연도   월       연월 수출입구분 수출입구분설명        물동량
0    2001   1  2001-01    IT    수입환적  122596.00
1    2001   2  2001-02    IT    수입환적   97114.00
2    2001   3  2001-03    IT    수입환적  134358.50
3    2001   4  2001-04    IT    수입환적  126275.00
4    2001   5  2001-05    IT    수입환적  120628.25
..    ...  ..      ...   ...     ...        ...
285  2024  10  2024-10    IT    수입환적  574791.75
286  2024  11  2024-11    IT    수입환적  571301.25
287  2024  12  2024-12    IT    수입환적  593917.50
288  2025   1  2025-01    IT    수입환적  635437.00
289  2025   2  2025-02    IT    수입환적  533568.75

[290 rows x 6 columns]>

In [ ]:
# 전체 물동량 중 환적 비율
전체환적비율 = (
    (df['수출환적'] + df['수입환적']) / (수출 + 수입 + 수출환적 + 수입환적)

TypeError: unsupported operand type(s) for /: 'str' and 'str'